# 02 — Revisione bottom-up

Il notebook revisiona le baseline top-down esistenti. Ricostruisce HLG′ dai soli LLG, calcola le cosine e raccoglie le decisioni documentali di Ministral, Granite e Gemma tramite un prompt globale per modello. Il quorum richiede due voti validi concordi su tre.

La bottom-up non esegue audit diretti degli LLG o diagnosi della fedeltà della ricostruzione. La qualità degli obiettivi generati resta affidata ai due critic originali. Le richieste vengono riunite e passate a generazione e reflection top-down originali sulle collezioni complete. Il bottom-up riprende il risultato soltanto dopo gli LLG e lo rivaluta nella successiva iterazione, anche se il punteggio finale è sotto soglia.

Ogni progetto ha un solo JSON in `output/top_down_bottom_up/runs/<batch>/datasets/`, con tutte le iterazioni. `results_summary.json` contiene tutti i dataset e lo stato del sistema.

Per calcolare le metriche esterne e confrontare questa esecuzione con tutti gli esperimenti, usare [04_evaluation_all_configurations.ipynb](04_evaluation_all_configurations.ipynb).

I riferimenti padre?figlio vengono recuperati solo se il riferimento completo non corrisponde e la descrizione identifica esattamente un unico HLG. La baseline resta intatta; aggiustamenti e riferimenti irrisolvibili sono registrati nello stato del ciclo. Non vengono creati figli per gli HLG che ne sono privi.


In [ ]:
from pathlib import Path
import json
import re
import sys
import time
from datetime import datetime

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

NOTEBOOK_DIR = PROJECT_ROOT / "notebook"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from groundtruth import ALL_GROUNDTRUTHS
import src.llm_clients as llm_clients
GROUNDTRUTH_NAMES = {item["name"] for item in ALL_GROUNDTRUTHS}

OUTPUT_ROOT = PROJECT_ROOT / "output"
TOP_DOWN_DIR = OUTPUT_ROOT / "top_down"
BOTTOM_UP_BATCHES_DIR = OUTPUT_ROOT / "top_down_bottom_up" / "runs"
TOP_DOWN_BOTTOM_UP_DIR = BOTTOM_UP_BATCHES_DIR / datetime.now().strftime("%Y%m%d_%H%M%S_%f")
DATASET_RESULTS_DIR = TOP_DOWN_BOTTOM_UP_DIR / "datasets"

for path in (
    TOP_DOWN_DIR,
    TOP_DOWN_BOTTOM_UP_DIR,
    DATASET_RESULTS_DIR,
):
    path.mkdir(parents=True, exist_ok=True)

# Run all registered datasets using their existing top-down baselines.
# Start execution manually from this notebook.
DATASETS_TO_RERUN = [item["name"] for item in ALL_GROUNDTRUTHS]

BASELINE_FILE_OVERRIDES = [
    TOP_DOWN_DIR
    / f"{name}_ZERO_SHOT_feedback_original_top_down_top_down.json"
    for name in DATASETS_TO_RERUN
]

# One global documentary review per model, after numerical similarity.

# Maximum number of feedback iterations. By default use the cycle constant.
GLOBAL_CYCLE_MAX_ITERATIONS_OVERRIDE = 25

# Timeout for local inference, including reasoning.
REQUEST_TIMEOUT_SECONDS = 600.0
API_MAX_RETRIES = 1

# Process baselines sequentially; local inference has no cloud rate limit.
DATASETS_PER_BATCH = 3
INTER_DATASET_DELAY_SECONDS = 0
INTER_BATCH_DELAY_SECONDS = 0

print("Project root:", PROJECT_ROOT)
print("Top-down inputs:", TOP_DOWN_DIR)
print("Top-down + bottom-up outputs:", TOP_DOWN_BOTTOM_UP_DIR)
print("Review: cosine similarity + one global documentary prompt per model")
print("Qwen generator/reconstructor (LM Studio locale):", llm_clients.QWEN_GENERATOR_MODEL)
print("Bottom-up critic ensemble (LM Studio):", llm_clients.configured_bottom_up_critic_models())
print("Bottom-up ensemble prompting: FEW_SHOT")
print("Generation repair critic (LM Studio):", llm_clients.configured_critic_model())
print("LM Studio endpoint:", llm_clients.LM_STUDIO_BASE_URL)

## 1. Individuazione degli output top-down

Il notebook usa tutti i file `*_top_down.json` presenti nella cartella della baseline, salvo override espliciti.

In [ ]:
if BASELINE_FILE_OVERRIDES:
    baseline_files = [Path(p) for p in BASELINE_FILE_OVERRIDES]
else:
    baseline_files = []
    for path in sorted(TOP_DOWN_DIR.glob("*_top_down.json")):
        with path.open("r", encoding="utf-8") as file:
            dataset_name = json.load(file).get("name")
        if dataset_name in GROUNDTRUTH_NAMES:
            baseline_files.append(path)

missing = [p for p in baseline_files if not p.exists()]
if missing:
    raise FileNotFoundError(f"Missing baseline files: {missing}")
if not baseline_files:
    raise FileNotFoundError(
        f"No *_top_down.json found in {TOP_DOWN_DIR}. "
        "Run 01_pipeline_execution_top_down_only.ipynb first."
    )

print(f"Found {len(baseline_files)} top-down baseline file(s):")
for p in baseline_files:
    print("-", p.name)

## 2. Caricamento dello stato iniziale

I file top-down contengono direttamente `structuredHighLevelGoals` e `structuredLowLevelGoals`. Il ciclo riusa quindi le relazioni padre-figlio realmente prodotte dalla pipeline top-down, senza mapping hard-coded. Le correzioni entrano nel top-down dalla generazione HLG Zero-shot con critic Few-shot originale. Solo dopo l'approvazione degli HLG il top-down genera gli LLG. I vecchi branch restano attivi fino al completamento del passaggio; non esiste una rigenerazione LLG separata dal bottom-up. Il feedback contiene solo i giudizi testuali dei due o tre evaluator concordi. Descrizione e attori restano quelli della baseline. La configurazione della baseline viene conservata come provenienza, ma non controlla le nuove generazioni del ciclo.

In [ ]:
from src.data_model import Actors, HighLevelGoals, LowLevelGoals
from src.examples.shot_learning import ShotPromptingMode

BOTTOM_UP_GENERATION_PROMPTING_MODE = ShotPromptingMode.ZERO_SHOT
BOTTOM_UP_GENERATION_EVALUATOR_ABLATION = False
BOTTOM_UP_GENERATION_CRITIC_FEW_SHOT_EXAMPLES = 4


def _infer_mode_from_name(path: Path) -> str:
    match = re.search(r"(?:^|_)(ZERO_SHOT|ONE_SHOT|FEW_SHOT)(?=_|$)", path.stem.upper())
    if not match:
        raise ValueError(f"Cannot infer prompting mode from {path.name}")
    return match.group(1)


def _stage_generation_config(
    config: dict,
    stage_name: str,
    baseline_file: Path,
) -> tuple[ShotPromptingMode, bool]:
    stage = config.get(stage_name) or {}
    mode_name = stage.get("promptingMode") or _infer_mode_from_name(
        baseline_file
    )
    try:
        mode = ShotPromptingMode[str(mode_name).upper()]
    except KeyError as exc:
        raise ValueError(
            f"Unsupported {stage_name} prompting mode {mode_name!r} in "
            f"{baseline_file.name}"
        ) from exc
    evaluator_ablation = not bool(stage.get("evaluator", True))
    return mode, evaluator_ablation


def load_top_down_state(baseline_file: Path):
    with baseline_file.open("r", encoding="utf-8") as f:
        payload = json.load(f)

    description = payload["description"]
    config = payload.get("pipelineConfiguration") or {}
    hlg_mode, hlg_evaluator_ablation = _stage_generation_config(
        config, "highLevelGoals", baseline_file
    )
    llg_mode, llg_evaluator_ablation = _stage_generation_config(
        config, "lowLevelGoals", baseline_file
    )
    structured_hlg = payload.get("structuredHighLevelGoals")
    structured_llg = payload.get("structuredLowLevelGoals")

    if structured_hlg is None or structured_llg is None:
        raise ValueError(
            f"{baseline_file.name} does not contain structured top-down goals. "
            "Run 01_pipeline_execution_top_down_only.ipynb first."
        )

    high_level_goals = HighLevelGoals.model_validate(structured_hlg)
    low_level_goals = LowLevelGoals.model_validate(structured_llg)

    return {
        "payload": payload,
        "description": description,
        "high_level_goals": high_level_goals,
        "actors": Actors.model_validate(payload["structuredActors"]) if payload.get("structuredActors") else None,
        "low_level_goals": low_level_goals,
        "source_hlg_prompting_mode": hlg_mode,
        "source_llg_prompting_mode": llg_mode,
        "source_hlg_llama_ablation": hlg_evaluator_ablation,
        "source_llg_llama_ablation": llg_evaluator_ablation,
        "prompting_mode": BOTTOM_UP_GENERATION_PROMPTING_MODE,
        "llama_ablation": BOTTOM_UP_GENERATION_EVALUATOR_ABLATION,
        "source_kind": "structured baseline fields",
        "mapped_input_file": None,
    }

## 3. Revisione e salvataggio compatto

Ogni dataset ha un solo JSON in `output/top_down_bottom_up/runs/<batch>/datasets/`, aggiornato durante la revisione. Contiene decisioni di ogni modello, maggioranza finale, modifiche effettive e HLG/LLG per tutte le iterazioni. Il checkpoint incompleto resta distinto dalle iterazioni concluse. Non vengono creati file separati per cosine o iterazioni.


In [ ]:
import importlib
import src.data_model as _data_model
import src.extraction.extractor as _extractor
import src.bottom_up.goal_reconstructor as _goal_reconstructor
import src.bottom_up.goal_similarity as _goal_similarity
import src.bottom_up.global_goal_evaluator as _global_goal_evaluator
import src.bottom_up.goal_cycle as _bottom_up_cycle
import src.self_critique.refine_response as _reflection
import src.bottom_up.result_storage as _result_storage
importlib.reload(_data_model)
importlib.reload(llm_clients)
importlib.reload(_extractor)
importlib.reload(_goal_reconstructor)
importlib.reload(_reflection)
importlib.reload(_goal_similarity)
importlib.reload(_global_goal_evaluator)
importlib.reload(_result_storage)
importlib.reload(_bottom_up_cycle)
from src.data_model import Actors, HighLevelGoals, LowLevelGoals
from src.bottom_up.goal_cycle import (
    run_global_goal_cycle,
    MAX_ITERATIONS,
)
from src.self_critique.refine_response import (
    MAX_ATTEMPTS,
    QUALITY_THRESHOLD,
)


def _close_client_safely(client):
    if client is None:
        return
    try:
        client.close()
    except Exception as exc:
        print(f"Warning while closing client: {type(exc).__name__}: {exc}")


def reset_model_clients():
    llm_clients.REQUEST_TIMEOUT_SECONDS = REQUEST_TIMEOUT_SECONDS
    llm_clients.API_MAX_RETRIES = API_MAX_RETRIES
    llm_clients.reset_model_clients()


def close_model_clients():
    _close_client_safely(getattr(llm_clients, "qwen_client", None))
    _close_client_safely(getattr(llm_clients, "ministral_client", None))
    llm_clients.qwen_client = None
    llm_clients.ministral_client = None


def bottom_up_model_configuration() -> dict:
    return {
        "generatorProvider": "LM Studio (local)",
        "generatorModel": llm_clients.QWEN_GENERATOR_MODEL,
        "evaluatorProvider": "LM Studio (local)",
        "consoleVerbosity": "detailed",
        "evaluatorResponseFormat": "json_schema",
        "critiqueCharacterTarget": _global_goal_evaluator.CRITIQUE_CHARACTER_TARGET,
        "evaluatorEndpoint": "/v1/chat/completions",
        "iterationOutputFiles": ["datasets/<progetto>.json", "results_summary.json"],
        "evaluatorModels": list(_global_goal_evaluator.BOTTOM_UP_EVALUATOR_MODELS),
        "evaluatorPromptingMode": _global_goal_evaluator.BOTTOM_UP_EVALUATOR_PROMPTING_MODE,
        "critic1FewShotExamples": _global_goal_evaluator.CRITIC1_FEW_SHOT_EXAMPLE_COUNT,
        "generationReflectionEvaluatorModel": llm_clients.configured_critic_model(),
        "evaluatorVotingQuorum": _global_goal_evaluator.EVALUATOR_VOTING_QUORUM,
        "critic1Policy": "ONE_GLOBAL_DOCUMENTARY_RESPONSE_WITH_FIXED_COSINE_MATCHES",
        "reconstructionRelationships": ["EQUIVALENT", "NOT_EQUIVALENT"],
        "similarity": _goal_similarity.similarity_configuration(),
        "structuralActionPriority": "REMOVALS_THEN_HLG_ENTRY_TOP_DOWN_HANDOFF",
        "repairPolicy": "original_top_down_start_at_hlg_then_llgs",
        "repairAcceptance": "hlg_critic_approval_then_llg_structural_validation_atomic_commit",
        "repairTraceField": "top_down_reflections",
        "repairFeedbackPolicy": "join_agreeing_feedback_critique_only",
        "parentReferencePolicy": "EXACT_REFERENCE_FIRST_THEN_UNIQUE_EXACT_DESCRIPTION",
        "groupingPolicy": "EXACT_EMBEDDED_PARENT_NAME_DESCRIPTION_AND_ACTOR",
        "branchConfirmationPolicy": "REEVALUATE_ALL_BRANCHES_EVERY_ITERATION",
        "evaluatorContextLength": llm_clients.LM_STUDIO_EVALUATOR_CONTEXT_LENGTH,
        "evaluatorVotingTemperature": _global_goal_evaluator.BOTTOM_UP_EVALUATOR_TEMPERATURE,
        "evaluatorVoteAttempts": _global_goal_evaluator.BOTTOM_UP_EVALUATOR_VOTE_ATTEMPTS,
        "datasetBatchSize": DATASETS_PER_BATCH,
        "interDatasetDelaySeconds": INTER_DATASET_DELAY_SECONDS,
        "interBatchDelaySeconds": INTER_BATCH_DELAY_SECONDS,
        "hlgGenerationPromptingMode": BOTTOM_UP_GENERATION_PROMPTING_MODE.name,
        "llgGenerationPromptingMode": BOTTOM_UP_GENERATION_PROMPTING_MODE.name,
        "hlgGenerationEvaluatorEnabled": not BOTTOM_UP_GENERATION_EVALUATOR_ABLATION,
        "llgGenerationEvaluatorEnabled": not BOTTOM_UP_GENERATION_EVALUATOR_ABLATION,
        "generationCriticFewShotExamples": BOTTOM_UP_GENERATION_CRITIC_FEW_SHOT_EXAMPLES,
        "maxIterations": (
            MAX_ITERATIONS
            if GLOBAL_CYCLE_MAX_ITERATIONS_OVERRIDE is None
            else GLOBAL_CYCLE_MAX_ITERATIONS_OVERRIDE
        ),
        "qualityThreshold": QUALITY_THRESHOLD,
        "maxReflectionAttempts": MAX_ATTEMPTS,
        "pipelinePolicyVersion": _global_goal_evaluator.PIPELINE_POLICY_VERSION,
        "evaluatorMaxCompletionTokens": llm_clients.BOTTOM_UP_CRITIC_MAX_TOKENS,
        "collectionRepairBudgetPolicy": "ONE_PASS_PER_GLOBAL_ITERATION_NO_SEPARATE_CAP",
    }


from src.bottom_up.result_storage import DatasetResults, dataset_filename, write_summary

def dataset_result_file(name):
    return DATASET_RESULTS_DIR / dataset_filename(name)


## 4. Esecuzione bottom-up

Ogni dataset viene processato in modo indipendente. Il JSON del dataset conserva la baseline, le decisioni, tutte le iterazioni e le collezioni finali. Il file top-down di partenza non viene mai sovrascritto.



In [ ]:
cycle_results = {}
cycle_errors = {}
max_iterations = (MAX_ITERATIONS if GLOBAL_CYCLE_MAX_ITERATIONS_OVERRIDE is None
                  else int(GLOBAL_CYCLE_MAX_ITERATIONS_OVERRIDE))


def wait_before_next_dataset(dataset_index, total_datasets):
    if dataset_index < total_datasets:
        time.sleep(INTER_BATCH_DELAY_SECONDS if dataset_index % DATASETS_PER_BATCH == 0
                   else INTER_DATASET_DELAY_SECONDS)


def run_profile(baseline_file, state):
    dataset_name = state["payload"]["name"]
    store = DatasetResults(dataset_result_file(dataset_name), state["payload"])
    store.start("bottom_up", configuration=bottom_up_model_configuration())
    started = time.perf_counter()
    try:
        reset_model_clients()
        result = run_global_goal_cycle(
            project_description=state["description"], actors=state["actors"],
            initial_high_level_goals=state["high_level_goals"],
            initial_low_level_goals=state["low_level_goals"], mode=state["prompting_mode"],
            evaluator_ablation=state["llama_ablation"], max_iterations=max_iterations,
            verbose=True, on_state=lambda cycle: store.update("bottom_up", cycle))
        store.complete("bottom_up", result)
        cycle_results[dataset_name] = result
    except BaseException as exc:
        interrupted = not isinstance(exc, Exception)
        message = f"{type(exc).__name__}: {exc}"
        store.fail("bottom_up", message, interrupted=interrupted)
        cycle_errors[dataset_name] = message
        if interrupted:
            raise
    finally:
        store.data["runs"]["bottom_up"]["elapsed_seconds"] = time.perf_counter() - started
        store.save()
        close_model_clients()
    print("Updated:", store.path)


def run_all_datasets():
    global TOP_DOWN_BOTTOM_UP_DIR, DATASET_RESULTS_DIR
    # A new batch for each invocation; previous results are never overwritten.
    TOP_DOWN_BOTTOM_UP_DIR = BOTTOM_UP_BATCHES_DIR / datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    DATASET_RESULTS_DIR = TOP_DOWN_BOTTOM_UP_DIR / "datasets"
    DATASET_RESULTS_DIR.mkdir(parents=True, exist_ok=False)
    print("New bottom-up batch:", TOP_DOWN_BOTTOM_UP_DIR)
    cycle_results.clear()
    cycle_errors.clear()
    files = []
    # Initialize the new batch before inference so stale results are not included.
    for path in baseline_files:
        try:
            baseline = json.loads(path.read_text(encoding="utf-8"))
        except Exception:
            baseline = {"name": path.stem}
        store = DatasetResults(dataset_result_file(baseline["name"]), baseline)
        store.data["runs"]["bottom_up"] = {"status": "not_started", "repetition": 1,
                                             "converged": None}
        store.save()
        files.append(store.path)
    write_summary(TOP_DOWN_BOTTOM_UP_DIR, files, planned_runs=len(baseline_files))
    try:
        for index, baseline_file in enumerate(baseline_files, 1):
            print(f"DATASET {index}/{len(baseline_files)}: {baseline_file.name}")
            try:
                state = load_top_down_state(baseline_file)
            except Exception as exc:
                document = json.loads(files[index-1].read_text(encoding="utf-8"))
                store = DatasetResults(files[index-1], document["baseline"])
                store.start("bottom_up")
                message = f"INPUT_LOAD_ERROR: {type(exc).__name__}: {exc}"
                store.fail("bottom_up", message)
                cycle_errors[document["name"]] = message
            else:
                run_profile(baseline_file, state)
            write_summary(TOP_DOWN_BOTTOM_UP_DIR, files, planned_runs=len(baseline_files))
            wait_before_next_dataset(index, len(baseline_files))
    finally:
        write_summary(TOP_DOWN_BOTTOM_UP_DIR, files, planned_runs=len(baseline_files))
    print("Summary:", TOP_DOWN_BOTTOM_UP_DIR / "results_summary.json")


## 4. Avvio

`run_all_datasets()` aggiorna i file dei dataset per la nuova esecuzione e produce `results_summary.json`, con tutti i risultati, le decisioni, i goal finali, gli errori e la convergenza. I risultati della run precedente vengono sostituiti nei nuovi file compatti; gli archivi storici non vengono cancellati.


In [ ]:
run_all_datasets()


## 5. Confronto strutturale rapido

Questa sezione mostra solo le cardinalità prima/dopo e lo stato di convergenza. **Precision, Recall e F1 non vengono calcolati qui**: sono volutamente delegati al notebook 04, che confronta il risultato bottom-up con la rispettiva baseline top-down.

In [ ]:
print("\nBottom-up execution summary")
print("---------------------------")
print(f"Inputs considered: {len(baseline_files)}")
print(f"Cycles completed: {len(cycle_results)}")
print(f"Cycles failed: {len(cycle_errors)}")
print(f"Bottom-up outputs (cosine + documentary review): {TOP_DOWN_BOTTOM_UP_DIR}")
print(f"Iteration traces: {DATASET_RESULTS_DIR}")

for dataset_name, result in cycle_results.items():
    print(
        f"- {dataset_name}: converged={result.converged}, "
        f"stop_reason={result.stop_reason}, "
        f"iterations={result.completed_iterations}, "
        f"final HLG={len(result.final_high_level_goals.goals)}, "
        f"final LLG={len(result.final_low_level_goals.low_level_goals)}"
    )
    if result.warnings:
        print("  warnings:", len(result.warnings), "(details in iteration JSON)")

if cycle_errors:
    print("\nErrors:")
    for dataset_name, error in cycle_errors.items():
        print(f"- {dataset_name}: {error}")